In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForSequenceClassification

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

import warnings
warnings.filterwarnings("ignore")

seed = 123

In [2]:
# Params
TESTING_LOCAL:bool = True

MODEL_NAME = "vinai/bertweet-base"
MAX_LEN = 3 if TESTING_LOCAL else 64
EPOCHS = 1 if TESTING_LOCAL else 5
BATCH_SIZE = 2 if TESTING_LOCAL else 32
FOLD = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# import data and check missing values
train = pd.read_csv("data/train.csv").sample(10) if TESTING_LOCAL else pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv").sample(3) if TESTING_LOCAL else pd.read_csv("data/test.csv")

# rename target col to labels
train = train.rename(columns={"target": "labels"})
test["labels"] = 0

# print missing
print("Train missing values:\n", train.isnull().sum())
print("\nTest missing values:\n", test.isnull().sum())

Train missing values:
 id          0
keyword     0
location    3
text        0
labels      0
dtype: int64

Test missing values:
 id          0
keyword     1
location    2
text        0
labels      0
dtype: int64


In [4]:
# clean text, store cleaned as col, "text_cleaned"
def clean_text(text_in:str):
    out = text_in
    out = re.sub("\n", " ", out) # strip multiline

    out = re.sub("B1G", "BIG", out) # strip multiline

    out = re.sub(r"http[^\s]+", " ", out) # strip http* link
    out = re.sub(r'--&gt;', '', out)
    out = out.replace(r'&lt;', '')
    out = out.replace(r'&gt;', '')
    out = out.replace(r'&amp;', '')
    out = re.sub(r"@\w+", "@USER", out) # @___ -> @USER
    out = re.sub(r"[" + re.escape('!"$%&()*+,-./:;<=>?[\\]^_`{|}~') + "]", " ", out) # most punctuation -> space
    out = re.sub(r"\d+", " ", out) # strip numbers
    out = re.sub(" ' ", " ", out) # most punctuation -> space

    # final space processing
    out = re.sub(r"\s+", " ", out) # multispace to 1 space
    out = re.sub(r"^\s+", "", out) # trim leading space
    out = re.sub(r"\s+$", "", out) # trim trailing space

    return out

train["text_cleaned"] = train["text"].apply(clean_text)
test["text_cleaned"] = test["text"].apply(clean_text)

# print before and after
_train_index = None
_test_index = None

# _train_index = 5803
# _test_index = 2125

_tmp_train = train.sample(1) if _train_index is None else train.loc[[_train_index]]
print("[Train] Row Index: ", _tmp_train.index[0])
print("[Train] Before clean: ", _tmp_train["text"].values[0])
print("[Train] After clean: ", _tmp_train["text_cleaned"].values[0])
print("===========================================================")
_tmp_test = test.sample(1) if _test_index is None else test.loc[[_test_index]]
print("[Test] Row Index: ", _tmp_test.index[0])
print("[Test] Before clean: ", _tmp_test["text"].values[0])
print("[Test] After clean: ", _tmp_test["text_cleaned"].values[0])

# clean tmp var
del _tmp_train, _tmp_test, _train_index, _test_index

[Train] Row Index:  5914
[Train] Before clean:  Update: The police seemed like nice enough people. I felt bad when I transformed and they screamed... Don't worry they're only unconscious.
[Train] After clean:  Update The police seemed like nice enough people I felt bad when I transformed and they screamed Don't worry they're only unconscious
[Test] Row Index:  451
[Test] Before clean:  Jersey's artist Wolfgangjoc pays homage to the late great Biggie Smalls by body bagging  his hypnotize beat.... http://t.co/x0jKnOxJHA
[Test] After clean:  Jersey's artist Wolfgangjoc pays homage to the late great Biggie Smalls by body bagging his hypnotize beat


In [5]:
# pytorch Dataset Class
import torch
from torch.utils.data import Dataset, DataLoader

class TweetData(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data["text_cleaned"].iloc[idx]
        labels = self.data["labels"].iloc[idx]

        encoded = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

In [6]:
# load berttweet tokenizer
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, Normalized=True) # Normalized to unify words

In [7]:
def train_epoch(train:DataLoader, model, optim:AdamW, sched:get_linear_schedule_with_warmup, lossfn):
    model.train()
    for batches in train:
        X = batches["input_ids"].to(DEVICE)
        X_att = batches["attention_mask"].to(DEVICE)
        Y = batches["labels"].to(DEVICE)

        output = model(input_ids=X, attention_mask=X_att)
        loss_value = lossfn(output.logits, Y)

        loss_value.backward()
        optim.step()
        sched.step()
        optim.step()
    return loss_value.item()

def predict_epoch(test:DataLoader, model):
    model.eval()
    out_logit = np.empty((0, 2))
    for batches in test:
        X = batches["input_ids"].to(DEVICE)
        X_att = batches["attention_mask"].to(DEVICE)

        logits = model(input_ids=X, attention_mask=X_att)
        out_logit = np.vstack([out_logit, logits.logits.detach().cpu().numpy()])
    return out_logit

In [8]:
oof_labels_prob_train = np.zeros(train.shape[0]) # softmax prob of labels=1
oof_labels_prob_test = np.zeros((test.shape[0], FOLD)) # softmax prob of labels=1 for each fold
loss_matrix = np.empty((FOLD, EPOCHS)) # [fold, epoch]

fold = StratifiedKFold(n_splits=FOLD, shuffle=True, random_state=123)
for ith_fold, (train_idx, eva_idx) in enumerate(fold.split(X=train, y=train["labels"])):
    _train = train.iloc[train_idx]
    _eval = train.iloc[eva_idx]

    _dataloader_train = DataLoader(TweetData(_train, bert_tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
    _dataloader_eva = DataLoader(TweetData(_eval, bert_tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
    _dataloader_test = DataLoader(TweetData(test, bert_tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

    _model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    _optimiser = AdamW(params=_model.parameters(), lr=1e-6)
    _scheduler = get_linear_schedule_with_warmup(_optimiser, num_warmup_steps=0, num_training_steps=len(_dataloader_train) * EPOCHS)
    loss_func = torch.nn.CrossEntropyLoss().to(DEVICE)

    # training
    for eth_epochs in range(EPOCHS):
        print("[START TRAINING] Fold {}/{} - Epoch {}/{}".format(ith_fold+1, FOLD, eth_epochs+1, EPOCHS))
        _loss_value = train_epoch(_dataloader_train, _model, _optimiser, _scheduler, loss_func)
        loss_matrix[ith_fold, eth_epochs] = _loss_value
        print("[END TRAINING] Fold {}/{} - Epoch {}/{} - Loss: {:.4f}".format(ith_fold+1, FOLD, eth_epochs+1, EPOCHS, _loss_value))

    # after train current fold
    print("[START OOF TRAIN] Fold {}/{}".format(ith_fold+1, FOLD))
    _eva_logits = predict_epoch(_dataloader_eva, _model)
    oof_labels_prob_train[eva_idx] = F.softmax(torch.tensor(_eva_logits, requires_grad=False).to(DEVICE), dim=1).detach().cpu().numpy()[:,1]
    print("[END OOF TRAIN] Fold {}/{}".format(ith_fold+1, FOLD))

    print("[START OOF TEST] Fold {}/{}".format(ith_fold+1, FOLD))
    _test_logits = predict_epoch(_dataloader_test, _model)
    oof_labels_prob_test[:, ith_fold] = F.softmax(torch.tensor(_test_logits, requires_grad=False).to(DEVICE), dim=1).detach().cpu().numpy()[:,1]
    print("[END OOF TEST] Fold {}/{}".format(ith_fold+1, FOLD))

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[START TRAINING] Fold 1/5 - Epoch 1/1
[END TRAINING] Fold 1/5 - Epoch 1/1 - Loss: 0.6648
[START OOF TRAIN] Fold 1/5
[END OOF TRAIN] Fold 1/5
[START OOF TEST] Fold 1/5
[END OOF TEST] Fold 1/5


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[START TRAINING] Fold 2/5 - Epoch 1/1
[END TRAINING] Fold 2/5 - Epoch 1/1 - Loss: 0.6728
[START OOF TRAIN] Fold 2/5
[END OOF TRAIN] Fold 2/5
[START OOF TEST] Fold 2/5
[END OOF TEST] Fold 2/5


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[START TRAINING] Fold 3/5 - Epoch 1/1
[END TRAINING] Fold 3/5 - Epoch 1/1 - Loss: 0.6073
[START OOF TRAIN] Fold 3/5
[END OOF TRAIN] Fold 3/5
[START OOF TEST] Fold 3/5
[END OOF TEST] Fold 3/5


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[START TRAINING] Fold 4/5 - Epoch 1/1
[END TRAINING] Fold 4/5 - Epoch 1/1 - Loss: 0.7031
[START OOF TRAIN] Fold 4/5
[END OOF TRAIN] Fold 4/5
[START OOF TEST] Fold 4/5
[END OOF TEST] Fold 4/5


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[START TRAINING] Fold 5/5 - Epoch 1/1
[END TRAINING] Fold 5/5 - Epoch 1/1 - Loss: 0.6567
[START OOF TRAIN] Fold 5/5
[END OOF TRAIN] Fold 5/5
[START OOF TEST] Fold 5/5
[END OOF TEST] Fold 5/5


In [9]:
# save all important variable
np.save('saved_var/oof_labels_prob_train.npy', oof_labels_prob_train)
np.save('saved_var/oof_labels_prob_test.npy', oof_labels_prob_test)
np.save('saved_var/loss_matrix.npy', loss_matrix)